In [42]:
import os
from pathlib import Path
import torch
from transformers import (AutoProcessor, AutoModelForSpeechSeq2Seq)
import soundfile as sf
from IPython.display import Audio
import numpy as np

In [43]:
DATASET_PATH = Path("/home/sgoyal/Projects/llm_asr_clarification/shared/datasets/amicorpus/train")
ASR_MODEL_NAME = "openai/whisper-tiny"
MEETING_NAME = "ES2005d"
MEETING_FOLDER = DATASET_PATH / MEETING_NAME
SAMPLING_RATE = 16_000

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

In [44]:
# AutoProcessor handles text tokenization AND audio feature extraction
processor = AutoProcessor.from_pretrained(ASR_MODEL_NAME)

# AutoModelForSpeechSeq2Seq handles the actual encoder-decoder network
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    ASR_MODEL_NAME,
    device_map=DEVICE,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

model.generation_config.language = "en"
model.generation_config.task = "transcribe"

In [45]:
audio_file_path = MEETING_FOLDER / "audio" / "ES2005d.Mix-Headset.wav"
waveform, _ = sf.read(audio_file_path) # waveform shape: (num_frames,)
waveform = waveform[SAMPLING_RATE * 60: SAMPLING_RATE * 120]

In [46]:
Audio(waveform, rate=SAMPLING_RATE)

In [53]:
noise_waveform = waveform + np.random.normal(loc=0.0, scale=0.002, size=waveform.shape)

In [54]:
Audio(noise_waveform, rate=SAMPLING_RATE)